# Data Masking

In [ ]:
import pandas as pd
import numpy as np
import hashlib
import re
import spacy


In [ ]:
!python -m spacy download en_core_web_lg
nlp = spacy.load('en_core_web_lg', disable=['parser', 'lemmatizer'])


In [ ]:
#masking config
MASKING_CONFIG = {
    # Amount thresholds
    'amount_keep_below': 200,        # keep exact value
    'amount_bucket_below': 10000,    # bucket into ranges
    'amount_drop_above': 10000,      # drop transaction entirely

    # Amount bucket edges
    'amount_bins':   [200, 500, 1000, 2000, 5000, 10000],
    'amount_labels': ['$200-500', '$500-1k', '$1k-2k', '$2k-5k', '$5k-10k'],

    # Columns to hash (+ save mapping)
    'hash_columns': ['transaction_id', 'user_id', 'account_id'],

    # Columns to apply full NER + regex
    'full_ner_columns': ['original_description'],

    # Columns to apply light NER (person names only)
    'light_ner_columns': ['cdr_biller_name', 'merchant_name', 'prop_ideas_merchant_name'],

    # Columns to drop entirely
    'drop_columns': [
        'transaction_date', 'is_detail_available', 'container',
        'properties', 'prop_ner_entity', 'prop_ner_entity_original',
        'prop_ideas_employer_entity', 'prop_memo',
        'prop_ideas_version', 'prop_ideas_enrich_steps', 'prop_ideas_categorised_at',
        'prop_ml_transfer_sub_category_key', 'frollo_merchant',
    ],

    # Column renames
    'rename_columns': {
        'external_id': 'provider_external_id',
    },

    # Regex patterns for full NER masking
    'regex_patterns': {
        'email': r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}',
        'phone': r'(?:\+61\s?|0)[2-9]\d{8}\b',
        'bsb':   r'\b\d{3}-\d{3}\b',
    },

    # Hash salt (change this to your own)
    'hash_salt': 'your-secret-salt-here',
}


In [ ]:
#3. Masking functions
# Whitelist: terms spaCy incorrectly tags as PERSON
NER_WHITELIST = {
    'CARD', 'VISA', 'EFTPOS', 'MASTERCARD', 'AMEX', 'DEBIT',
    'PERTH', 'SYDNEY', 'MELBOURNE', 'BRISBANE', 'ADELAIDE',
    'HOBART', 'DARWIN', 'CANBERRA', 'GOLD COAST',
    'ALBERT PARK', 'KARRINYUP', 'WOOLLOONGABBA', 'MAROOCHYDORE',
    'SURRY HILLS', 'SURRY', 'BROOKLYN PARK', 'WYONG', 'WAHROONGA',
    'HARRISDALE', 'ALBURY', 'MACQUARIEPARK',
    'PTY', 'LTD', 'INC', 'BPAY', 'PAYPAL', 'ROYAL',
    'PETER', "PETER'S", 'CHURCH', 'SDA', 'SEED',
    'ZIP', 'ZIP.CO',
}

# Keywords that indicate a person name likely follows
TRANSFER_KEYWORDS = {'TO', 'FROM', 'PAYMENT', 'TRANSFER', 'TFR'}

#  Keywords that indicate description is a business/merchant, not a person
BUSINESS_KEYWORDS = {
    # Venue types
    'CAFE', 'BAKERY', 'BAKERS', 'RESTAURANT', 'BAR', 'PUB', 'HOTEL', 'MOTEL',
    'PHARMACY', 'CHEMIST', 'MEDICAL', 'DENTAL', 'PHYSIO',
    'AUTO', 'AUTOMOTIVE', 'MOTORS', 'TYRES', 'BATTERIES',
    'ELECTRICAL', 'PLUMBING', 'ROOFING', 'PAINTING',
    'SALON', 'BARBER', 'BEAUTY', 'SPA', 'NAILS',
    'GYM', 'FITNESS', 'YOGA', 'PILATES',
    'SCHOOL', 'COLLEGE', 'UNIVERSITY',
    'CHURCH', 'MOSQUE', 'TEMPLE',

    # Retail
    'STORE', 'STORES', 'SHOP', 'MART', 'MARKET', 'IGA', 'SUPA',
    'NEWSAGENT', 'KIOSK', 'OUTLET', 'WAREHOUSE',
    'FLORIST', 'JEWELLER', 'BUTCHER', 'GROCER',

    # Corporate
    'PTY', 'LTD', 'INC', 'CORP', 'GROUP', 'HOLDINGS',
    'AUSTRALIA', 'SERVICES', 'SOLUTIONS', 'TRADING',

    # Payment platforms
    'PAYPAL', 'BPAY', 'AFTERPAY', 'ZIPPAY',

    # Transaction prefixes
    'EFTPOS', 'VISA', 'MASTERCARD', 'AMEX', 'DEBIT',
    'PURCHASE', 'CARD',
}


def hash_value(val, salt=MASKING_CONFIG['hash_salt']):
    """One-way hash. Same input always gives same output."""
    if pd.isna(val):
        return None
    return hashlib.sha256(f"{salt}:{val}".encode()).hexdigest()[:16]


def is_likely_person(ent, full_text):
    """
    Determine if a spaCy PERSON entity is likely a real person name.
    Rules:
      1. Skip if whitelisted
      2. Skip if any business keyword appears in the description
      3. Only mask if preceded by a transfer keyword (TO, FROM, PAYMENT, etc.)
    """
    if ent.text.upper() in NER_WHITELIST:
        return False

    # Skip if any business keyword is present (word-level match, no substrings)
    desc_words = set(full_text.upper().split())
    if desc_words & BUSINESS_KEYWORDS:
        return False

    # Only mask if preceded by a transfer keyword (within last 3 words)
    prefix = full_text[:ent.start_char].upper().split()
    after_transfer = any(kw in prefix[-3:] for kw in TRANSFER_KEYWORDS) if prefix else False

    return after_transfer
    

def mask_full_ner(text, config=MASKING_CONFIG):
    """Full NER + regex masking for free-text fields like original_description."""
    if pd.isna(text):
        return text
    s = str(text)

    # Regex pass first
    for name, pattern in config['regex_patterns'].items():
        s = re.sub(pattern, f'[{name.upper()}]', s)

    # spaCy NER pass — only redact PERSON if it looks like a real name
    doc = nlp(s)
    for ent in reversed(doc.ents):
        if ent.label_ == 'PERSON' and is_likely_person(ent, s):
            s = s[:ent.start_char] + '[PERSON]' + s[ent.end_char:]

    return s


def mask_light_ner(text):
    """Light NER — redact person names only from merchant/biller fields."""
    if pd.isna(text):
        return text
    s = str(text)
    doc = nlp(s)
    for ent in reversed(doc.ents):
        if ent.label_ == 'PERSON' and is_likely_person(ent, s):
            s = s[:ent.start_char] + '[PERSON]' + s[ent.end_char:]
    return s


def derive_amount_features(df, col='amount', config=MASKING_CONFIG):
    """Derive amount features, bucket, and drop original."""
    amt = df[col].astype(float).abs()

    df = df.assign(
        amount_is_round=(amt % 1 == 0),
        amount_is_round_10=(amt % 10 == 0),
        amount_decimal_part=(amt % 1).round(2),
        amount_ends_in_95_98_99=amt.apply(lambda x: round(x % 1, 2) in (0.95, 0.98, 0.99)),
        amount_is_negative=(df[col] < 0),
    )

    # Drop transactions above threshold
    df = df[amt <= config['amount_drop_above']].copy()
    amt = df[col].astype(float).abs()

    # Hybrid: keep exact for small, bucket for medium
    keep = config['amount_keep_below']
    df['amount_masked'] = np.where(
        amt < keep,
        amt.round(2).astype(str),
        pd.cut(amt, bins=config['amount_bins'], labels=config['amount_labels'], right=True)
    )

    df = df.drop(columns=[col])
    return df


def apply_masking(df, config=MASKING_CONFIG):
    """Full masking pipeline."""
    df = df.copy()
    n_before = len(df)

    # 1. Rename columns
    df = df.rename(columns=config['rename_columns'])

    # 2. Hash ID columns + save mapping
    mapping_parts = {}
    for col in config['hash_columns']:
        if col in df.columns:
            mapping_parts[col] = df[[col]].drop_duplicates().copy()
            mapping_parts[col][f'{col}_hashed'] = mapping_parts[col][col].apply(hash_value)
            df[col] = df[col].apply(hash_value)

    mapping = None
    if mapping_parts:
        mapping = {k: v for k, v in mapping_parts.items()}

    # 3. Full NER + regex
    for col in config['full_ner_columns']:
        if col in df.columns:
            df[col] = df[col].apply(mask_full_ner)

    # 4. Light NER
    for col in config['light_ner_columns']:
        if col in df.columns:
            df[col] = df[col].apply(mask_light_ner)

    # 5. Amount features + masking
    if 'amount' in df.columns:
        df = derive_amount_features(df, config=config)

    # 6. Drop excluded columns (only those that exist)
    drop_cols = [c for c in config['drop_columns'] if c in df.columns]
    df = df.drop(columns=drop_cols)

    n_after = len(df)
    print(f"Rows: {n_before:,} → {n_after:,} (dropped {n_before - n_after:,} over ${config['amount_drop_above']:,})")

    return df, mapping


In [ ]:
df_sampled = pd.read_parquet("data/sample_data3.parquet")

In [ ]:
#4 Run masking on one provider to test
TEST_PROVIDER = 'ANZ'  # ← change to test different providers

df_test = df_sampled[df_sampled['provider_name'] == TEST_PROVIDER].copy()
df_masked_test, mapping_test = apply_masking(df_test)

print(f"\nColumns: {df_masked_test.columns.tolist()}")
print(f"Shape: {df_masked_test.shape}")

In [ ]:
#5: Before/after masking comparison

# Show the dropped high-value transactions
df_dropped = df_test[df_test['amount'].astype(float).abs() > MASKING_CONFIG['amount_drop_above']]
print(f"=== Dropped Transactions ({len(df_dropped)}) ===")
print(df_dropped[['original_description', 'amount', 'base_category_key']].to_string(index=False))

# Compare descriptions: before vs after
orig = df_test.loc[df_masked_test.index, 'original_description']
masked = df_masked_test['original_description']

changed = orig != masked
print(f"\n=== Description Masking Summary ===")
print(f"Total:     {len(masked)}")
print(f"Changed:   {changed.sum()}")
print(f"Unchanged: {(~changed).sum()}")

# Show only changed rows side-by-side
if changed.sum() > 0:
    print(f"\n=== Changed Descriptions (showing first 20) ===\n")
    for o, m in zip(orig[changed].head(20), masked[changed].head(20)):
        print(f"  BEFORE: {o}")
        print(f"  AFTER:  {m}\n")

In [ ]:
#6 Check amount features
print("=== Amount Feature Distribution ===\n")
print(f"amount_masked value counts:\n{df_masked_test['amount_masked'].value_counts().head(15)}\n")
print(f"amount_is_round:           {df_masked_test['amount_is_round'].mean():.1%}")
print(f"amount_is_round_10:        {df_masked_test['amount_is_round_10'].mean():.1%}")
print(f"amount_ends_in_95_98_99:   {df_masked_test['amount_ends_in_95_98_99'].mean():.1%}")
print(f"amount_is_negative:        {df_masked_test['amount_is_negative'].mean():.1%}")

In [ ]:
#7: Check hash mapping
print("=== Hash Mapping Sample ===\n")
for col, m in mapping_test.items():
    print(f"{col}:")
    print(m.head(5).to_string(index=False))
    print()


In [ ]:
#8: Check NER hits on merchant fields
for col in MASKING_CONFIG['light_ner_columns']:
    if col in df_masked_test.columns:
        hits = df_masked_test[df_masked_test[col].str.contains(r'\[PERSON\]', na=False)]
        print(f"{col}: {len(hits)} rows with [PERSON] redaction")
        if len(hits) > 0:
            print(hits[col].head(10).to_string(index=False))
        print()

In [ ]:
#9: Happy? Run on full dataset
df_masked, mapping_full = apply_masking(df_sampled)

print(f"\nFinal shape: {df_masked.shape}")
print(f"Final columns:\n{df_masked.columns.tolist()}")

In [ ]:
#10: Final validation
print("=== Final Dataset Summary ===\n")

# No real IDs remain
for col in ['transaction_id', 'user_id', 'account_id']:
    if col in df_masked.columns:
        sample = df_masked[col].dropna().iloc[0] if df_masked[col].notna().any() else 'N/A'
        print(f"{col} sample: {sample}  (should be hash)")

# No raw amounts remain
assert 'amount' not in df_masked.columns, "ERROR: raw amount column still present!"
print(f"\namount_masked distribution:\n{df_masked['amount_masked'].value_counts()}\n")

# No excluded columns remain
for col in MASKING_CONFIG['drop_columns']:
    assert col not in df_masked.columns, f"ERROR: {col} still present!"
print("✅ All excluded columns removed")

# Description spot-check for any obvious PII leaks
print(f"\nRandom masked descriptions:")
print(df_masked['original_description'].sample(10, random_state=42).to_string(index=False))

In [ ]:
#11: Export
OUTPUT_PATH = './data/masked_transaction_dataset.parquet'
MAPPING_PATH = './data/id_mapping_SECURE.parquet'

df_masked.to_parquet(OUTPUT_PATH, index=False)
print(f"Dataset saved: {OUTPUT_PATH} ({len(df_masked):,} rows)")

# Save mapping — KEEP THIS SECURE
mapping_dfs = []
for col, m in mapping_full.items():
    mapping_dfs.append(m)
mapping_combined = pd.concat(mapping_dfs, axis=1)
mapping_combined.to_parquet(MAPPING_PATH, index=False)
print(f"Mapping saved: {MAPPING_PATH} — ⚠️ KEEP SECURE, DO NOT SHARE")